In [1]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments
from datasets import load_dataset
import datasets
import tqdm
import os

In [2]:
version = "SQuAD_2.0"

train_path = os.path.join(os.getcwd(), "Datasets", version, "train-00000-of-00001.parquet")
val_path = os.path.join(os.getcwd(), "Datasets", version, "validation-00000-of-00001.parquet")

dataset = load_dataset("parquet", data_files={'train': train_path, 'val': val_path})

# squad = datasets.Dataset(squad["train"][:5000])

In [4]:
print(dataset['train'][0])

{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}


In [3]:
squad= dataset["train"].select(range(5000))
squad = squad.train_test_split(test_size=0.2)

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [5]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    # inputs contain tokenised questions and context in same list, separated by [SEP] token
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        # accounts for questions with no answer and use CLS token
        if len(answer["answer_start"]) == 0:
            start_char = 0
            end_char = 0
        else:
            start_char = answer["answer_start"][0]
            end_char = answer["answer_start"][0] + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        # At the question, sequence id = 0, for context sequence id = 1
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [58]:
answers = squad["train"]
answer = answers[10]
print(answer)

{'id': '5a8dbd49df8bba001a0f9bb6', 'title': 'The_Legend_of_Zelda:_Twilight_Princess', 'context': 'A CD containing 20 musical selections from the game was available as a GameStop preorder bonus in the United States; it is included in all bundles in Japan, Europe, and Australia.[citation needed]', 'question': 'How many tracks were recorded on the post order CD?', 'answers': {'text': [], 'answer_start': []}}


In [6]:
tokenized_squad = squad.map(preprocess_function, batched=True, remove_columns=squad["train"].column_names)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
ex = tokenized_squad["train"][10]
tokens = tokenizer.convert_ids_to_tokens(ex["input_ids"])
print(tokens)
print(f"start {ex["start_positions"]}")
print(f"end {ex["end_positions"]}")

['[CLS]', 'filming', 'closed', 'which', 'two', 'bridges', 'in', 'london', '?', '[SEP]', 'a', 'brief', 'shoot', 'at', 'london', "'", 's', 'city', 'hall', 'was', 'filmed', 'on', '18', 'april', '2015', ',', 'while', 'mendes', 'was', 'on', 'location', '.', 'on', '17', 'may', '2015', 'filming', 'took', 'place', 'on', 'the', 'thames', 'in', 'london', '.', 'stunt', 'scenes', 'involving', 'craig', 'and', 'se', '##yd', '##oux', 'on', 'a', 'speed', '##boat', 'as', 'well', 'as', 'a', 'low', 'flying', 'helicopter', 'near', 'westminster', 'bridge', 'were', 'shot', 'at', 'night', ',', 'with', 'filming', 'temporarily', 'closing', 'both', 'westminster', 'and', 'lamb', '##eth', 'bridges', '.', 'scenes', 'were', 'also', 'shot', 'on', 'the', 'river', 'near', 'mi', '##6', "'", 's', 'headquarters', 'at', 'va', '##ux', '##hall', 'cross', '.', 'the', 'crew', 'returned', 'to', 'the', 'river', 'less', 'than', 'a', 'week', 'later', 'to', 'film', 'scenes', 'solely', 'set', 'on', 'westminster', 'bridge', '.', 'th

In [8]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [9]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
output_name = "test_bert"
output_dir = os.path.join(os.getcwd(), "models", output_name)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.320572
2,2.775300,1.821562
3,2.775300,1.770474


TrainOutput(global_step=750, training_loss=2.335025716145833, metrics={'train_runtime': 372.7868, 'train_samples_per_second': 32.19, 'train_steps_per_second': 2.012, 'total_flos': 1175877900288000.0, 'train_loss': 2.335025716145833, 'epoch': 3.0})

In [ ]:
from transformers import pipeline

final_checkpoint = os.path.join(output_dir, "checkpoint-750")

question_answerer = pipeline("question-answering", model=final_checkpoint)

Device set to use cuda:0


In [29]:
def get_answer_existence_list(dataset):
    """
    Creates a list with 0 for examples with no answer and 1 for examples with an answer.

    Args:
        dataset: A Hugging Face Dataset object (e.g., squad["test"]).

    Returns:
        A list of 0s and 1s representing answer existence.
    """
    answer_existence_list = []
    for example in dataset:
        answers = example["answers"]
        if not answers["text"]:
            answer_existence_list.append(0)  # No answer
        elif len(answers["text"][0]) == 0:
            answer_existence_list.append(0) # No answer
        else:
            answer_existence_list.append(1)  # Answer exists
    return answer_existence_list

# Assuming 'squad' is your loaded dataset
answer_list = get_answer_existence_list(squad["test"])

In [121]:
test = squad["test"][400]
test

{'id': '56bf7cb63aeaaa14008c9678',
 'title': 'Beyoncé',
 'context': 'The group changed their name to Destiny\'s Child in 1996, based upon a passage in the Book of Isaiah. In 1997, Destiny\'s Child released their major label debut song "Killing Time" on the soundtrack to the 1997 film, Men in Black. The following year, the group released their self-titled debut album, scoring their first major hit "No, No, No". The album established the group as a viable act in the music industry, with moderate sales and winning the group three Soul Train Lady of Soul Awards for Best R&B/Soul Album of the Year, Best R&B/Soul or Rap New Artist, and Best R&B/Soul Single for "No, No, No". The group released their multi-platinum second album The Writing\'s on the Wall in 1999. The record features some of the group\'s most widely known songs such as "Bills, Bills, Bills", the group\'s first number-one single, "Jumpin\' Jumpin\'" and "Say My Name", which became their most successful song at the time, and woul

In [122]:
question = test["question"]
context = test["context"]
p_answer=question_answerer(question=question, context=context)
p_answer

{'score': 0.2951500415802002,
 'start': 85,
 'end': 99,
 'answer': 'Book of Isaiah'}

In [63]:
t_tokenizer = AutoTokenizer.from_pretrained(final_checkpoint)

In [133]:
t_inputs = tokenizer(question, context, return_tensors="pt")
len(t_inputs["input_ids"][0])

346

In [65]:
t_model = AutoModelForQuestionAnswering.from_pretrained(final_checkpoint)

In [124]:
import torch

with torch.no_grad():
    t_outputs = t_model(**t_inputs)

In [113]:
t_outputs.start_logits

tensor([[-2.9806, -4.1531, -5.1184, -5.0499, -5.0402, -5.4774, -3.9966, -5.1607,
         -5.5274, -4.7183,  3.7281, -3.8475, -4.4048, -4.4757, -3.5022, -1.7482,
         -0.7874, -3.2201, -3.1976, -0.1903, -2.2261, -1.7298, -5.2558, -2.1698,
         -3.2842,  3.2034, -1.2236, -1.7937,  0.4292, -3.6452,  0.2981, -5.4241,
         -2.8997, -2.4639, -2.1468, -2.5132,  3.6704, -2.8263, -2.3543, -1.4529,
          2.8975, -1.4994, -3.7840, -5.2232, -2.1269, -1.1031, -4.1094, -4.9593,
         -4.2037, -3.3491, -4.1632, -4.0477,  2.4803, -3.7393, -3.8543, -5.0478,
         -5.6346, -4.8425, -3.2661, -4.8105, -4.5557, -4.5499, -4.7942, -5.2923,
         -5.3573, -5.2490, -4.4241, -5.4902, -5.0179, -4.7241, -3.6101, -5.5944,
         -5.3577, -4.7967, -3.6758, -5.3838, -5.5690, -4.9882, -4.1969, -5.2783,
         -5.9518, -5.2506, -4.0209, -4.5720, -4.8744, -1.3275, -4.6164, -5.5771,
         -5.1439, -4.7695, -4.8249, -5.1923, -4.7901, -3.9248, -5.5101, -5.2822,
         -5.3093, -4.7550, -

In [125]:
import torch.nn.functional as func

start_probs = func.softmax(t_outputs.start_logits, dim=-1).squeeze()
end_probs = func.softmax(t_outputs.end_logits, dim=-1).squeeze()

In [126]:
answer_start_index = t_outputs.start_logits.argmax()
answer_end_index = t_outputs.end_logits.argmax()

In [127]:
start = start_probs.argmax()
end = end_probs.argmax()
start_probs[start] * end_probs[end]

print(start)
print(end)

tensor(39)
tensor(41)


In [128]:
predict_answer_tokens = t_inputs.input_ids[0, answer_start_index : answer_end_index + 1]
t_answer=tokenizer.decode(predict_answer_tokens)
t_answer

'book of isaiah'

In [147]:
t_input = tokenized_squad["test"][0]

In [ ]:
with torch.no_grad():
    t_outputs = t_model(**t_input)

AttributeError: 'dict' object has no attribute 'long'

In [142]:
start_probs = func.softmax(t_outputs.start_logits, dim=-1).squeeze()
end_probs = func.softmax(t_outputs.end_logits, dim=-1).squeeze()

start_index = start_probs.argmax()
end_index = end_probs.argmax()

In [ ]:
predict_tokens = t_inputs.input_ids[0, start_index:end_index+1]
predicted_answer = tokenizer.decode(predict_tokens)

score = start_probs[start_index] * end_probs[end_index]
predicted_answer

'book of isaiah'